In [1]:
%pip install ipykernel
%pip install opencv-python 
%pip install matplotlib
%pip install matplotlib tk
%pip install plotly nbformat
%pip install pyqtgraph PyOpenGL PyQt5


Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import cv2 as cv  
import matplotlib.pyplot as plt
import numpy as np
from matplotlib import cm, colors
import matplotlib

import os
import sys
from collections import deque
import time


#real time graphs
import pyqtgraph as pg
import pyqtgraph.opengl as gl
from pyqtgraph.Qt import QtCore #TIMER


In [3]:
try:
    get_ipython().run_line_magic("matplotlib", "tk")   # notebook: 3D map opens as its own live window
except NameError:
    matplotlib.use(os.environ.get("MPLBACKEND", "TkAgg"))   # plain script
import matplotlib.pyplot as plt
 
# ======================================================================
# SETTINGS
# ======================================================================
CAMERA_INDEX = 0        # which camera. If it won't open, the other indexes are tried automatically
CAMERA_BACKEND = None   # Windows: set to cv.CAP_DSHOW if the camera won't open
SEAM = 98               # columns cropped off the left of each frame
WINDOW = "TC001"
DISPLAY_SCALE = 3       # enlarge the camera window (the thermal image is small)
 
HISTORY = 80            # time rows kept in the 3D map
COL_STEP = 4            # keep every Nth column (higher = faster 3D redraw)
ROW_MODE = "colmax"     # "scanline" (middle row) | "colmean" | "colmax" (hottest per column)
DRAW_EVERY = 2          # redraw the 3D map every N camera frames
MAX_READ_FAILS = 30     # failed reads in a row tolerated before giving up
 
 
# ======================================================================
# CAMERA
# ======================================================================
def _try_open(index):
    """Open a camera and confirm it really delivers frames. Returns the capture or None."""
    if CAMERA_BACKEND is None:
        cap = cv.VideoCapture(index)
    else:
        cap = cv.VideoCapture(index, CAMERA_BACKEND)
    if cap.isOpened():
        ok, _ = cap.read()
        if ok:
            return cap
    cap.release()
    return None
 
 
def find_cameras(max_index=6):
    """List the camera indexes that work, with frame shapes:  print(find_cameras())"""
    found = []
    for i in range(max_index):
        cap = _try_open(i)
        if cap is not None:
            ok, frame = cap.read()
            found.append((i, frame.shape if ok else None))
            cap.release()
    return found
 
 
def open_camera(index):
    """Connect to the camera (falling back to the first working one) and print its properties."""
    cap = _try_open(index)
    if cap is None:
        found = find_cameras()
        if not found:
            raise RuntimeError(
                "No camera could be opened. Close anything else using it (other notebook "
                "cells, apps), restart the kernel, or set CAMERA_BACKEND = cv.CAP_DSHOW on Windows.")
        fallback = found[0][0]
        print(f"Camera {index} did not open. Working cameras: {found}. Using camera {fallback}; "
              "set CAMERA_INDEX to pick the thermal one.")
        cap = _try_open(fallback)
    print("Frame Width:", cap.get(cv.CAP_PROP_FRAME_WIDTH))
    print("Frame Height:", cap.get(cv.CAP_PROP_FRAME_HEIGHT))
    print("FPS:", cap.get(cv.CAP_PROP_FPS))
    return cap
 
 
def read_gray(cap):
    """Grab one frame, crop the seam, return 8-bit grayscale (None if no frame)."""
    ok, frame = cap.read()
    if not ok:
        return None
    if frame.shape[1] <= SEAM:
        raise RuntimeError(f"Frame is only {frame.shape[1]} px wide (shape {frame.shape}), "
                           f"so SEAM={SEAM} crops everything. Lower SEAM or pick another camera.")
    crop = frame[:, SEAM:]
    return crop if crop.ndim == 2 else cv.cvtColor(crop, cv.COLOR_BGR2GRAY)
 
 
# ======================================================================
# THERMAL MAP DATA
# ======================================================================
class ThermalHistory:
    """Rolling buffer: each frame is reduced to one row of signal across the image."""
 
    def __init__(self, width):
        self.x = np.arange(0, width, COL_STEP)
        self.rows = deque([np.zeros(len(self.x), np.float32)] * HISTORY, maxlen=HISTORY)
        self.grid_x, self.grid_y = np.meshgrid(self.x, np.arange(HISTORY))
 
    def push(self, gray):
        values = gray.astype(np.float32)
        if ROW_MODE == "scanline":
            row = values[values.shape[0] // 2]
        elif ROW_MODE == "colmean":
            row = values.mean(axis=0)
        else:
            row = values.max(axis=0)
        self.rows.append(row[::COL_STEP])      # oldest row drops off automatically
 
    def surface(self):
        return np.array(self.rows)             # shape (time, x)
 
 
# ======================================================================
# DISPLAY
# ======================================================================
def hotspot_display(feed, gray, scale=1):
    """Crosshair and value on the hottest spot of the live camera feed."""
    _, peak, _, (hx, hy) = cv.minMaxLoc(cv.GaussianBlur(gray, (5, 5), 0))
    px, py = int(hx * scale + scale // 2), int(hy * scale + scale // 2)
    cv.drawMarker(feed, (px, py), (255, 255, 255), cv.MARKER_CROSS, 14, 1)
    cv.putText(feed, f"{peak:.0f}", (px + 8, max(py - 8, 12)),
               cv.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv.LINE_AA)
    return feed
 
 
class Map3D:
    """The 3D window: X = image column, Y = time, Z / colour = signal."""
 
    def __init__(self, history):
        self.history = history
        if "inline" in matplotlib.get_backend().lower():
            print("WARNING: matplotlib is in inline mode, so the 3D window can't update live. "
                  "Run  %matplotlib tk  in a cell above, then restart the kernel if needed.")
        plt.ion()
        self.fig = plt.figure("Thermal 3D Map", figsize=(8, 6))
        self.ax = self.fig.add_subplot(111, projection="3d")
 
    def draw(self, z):
        """Redraw from the (time, x) array. Returns False if the window was closed."""
        h, ax = self.history, self.ax
        ax.clear()
        ax.plot_surface(h.grid_x, h.grid_y, z, cmap="inferno", vmin=0, vmax=255,
                        linewidth=0, antialiased=False, rcount=HISTORY, ccount=len(h.x))
        self._mark_hotspot(z)
        ax.set_xlabel("Image X")
        ax.set_ylabel("Time (frames)")
        ax.set_zlabel("Signal")
        ax.set_zlim(0, 255)
        ax.view_init(elev=45, azim=-70)
        plt.pause(0.001)                       # lets the window refresh
        return plt.fignum_exists(self.fig.number)
 
    def _mark_hotspot(self, z):
        newest = z[-1]
        i = int(np.argmax(newest))
        x, y, v = self.history.x[i], HISTORY - 1, newest[i]
        self.ax.scatter([x], [y], [v], color="cyan", s=60, depthshade=False)
        self.ax.text(x, y, v + 10, f"{v:.0f}", ha="center")
 
 
# ======================================================================
# MAIN LOOP
# ======================================================================
def main():
    cap = open_camera(CAMERA_INDEX)
    cv.namedWindow(WINDOW)
    history = map3d = None
    count = fails = 0
    check_window = False                       # decided once the window exists
    reason = "interrupted or error (see traceback)"
 
    try:
        while True:
            gray = read_gray(cap)
            if gray is None:                   # dropped frame: wait briefly and retry
                fails += 1
                if fails >= MAX_READ_FAILS:
                    reason = "camera stopped delivering frames"
                    break
                cv.waitKey(10)
                continue
            fails = 0
            count += 1
 
            if history is None:                # size the map from the first real frame
                print("Frame after SEAM crop:", gray.shape)
                history = ThermalHistory(gray.shape[1])
                map3d = Map3D(history)
 
            # live camera feed (enlarged, with hotspot marker)
            feed = cv.applyColorMap(gray, cv.COLORMAP_INFERNO)
            if DISPLAY_SCALE != 1:
                feed = cv.resize(feed, None, fx=DISPLAY_SCALE, fy=DISPLAY_SCALE,
                                 interpolation=cv.INTER_CUBIC)
            cv.imshow(WINDOW, hotspot_display(feed, gray, DISPLAY_SCALE))
 
            # 3D map (update_frame)
            history.push(gray)
            if count % DRAW_EVERY == 0 and not map3d.draw(history.surface()):
                reason = "3D map window closed (or its backend can't stay open)"
                break
 
            key = cv.waitKey(1) & 0xFF
            if key in (ord("q"), 27):
                reason = "q / Esc pressed"
                break
 
            # Some OpenCV builds never report window visibility; only trust it if it
            # says "visible" once the window has had time to appear.
            if count == 5:
                check_window = cv.getWindowProperty(WINDOW, cv.WND_PROP_VISIBLE) >= 1
            if check_window and cv.getWindowProperty(WINDOW, cv.WND_PROP_VISIBLE) < 1:
                reason = "camera window closed"
                break
    finally:                                   # always free the camera, even on Ctrl-C / errors
        cap.release()
        cv.destroyAllWindows()
        for _ in range(4):
            cv.waitKey(1)                      # lets the window actually close in notebooks
        plt.close("all")
        print("Stopped:", reason, f"(after {count} frames)")
 
 
if __name__ == "__main__":
    main()

Frame Width: 484.0
Frame Height: 512.0
FPS: 25.0
Frame after SEAM crop: (512, 386)
Stopped: camera window closed (after 426 frames)


In [4]:
#Bounding box Code here and color bar 